Beyond Azure SQL and Cosmos DB, Azure offers a set of specialised managed database services:

| Service | AWS equivalent | Type |
|---|---|---|
| **Azure Cache for Redis** | Amazon ElastiCache (Redis) | In-memory cache / data store |
| **Azure Database for PostgreSQL** | Amazon RDS for PostgreSQL / Aurora PostgreSQL | Managed PostgreSQL |
| **Azure Database for MySQL** | Amazon RDS for MySQL / Aurora MySQL | Managed MySQL |
| **Azure Database for MariaDB** | Amazon RDS for MariaDB | Managed MariaDB |

This topic covers Redis in depth — the most commonly used caching layer in Azure architectures — and then surveys the open-source relational database services.

## Azure Cache for Redis

**Azure Cache for Redis** is a fully managed, in-memory data store based on the open-source Redis project. It provides sub-millisecond data access for caching, session storage, pub/sub messaging, leaderboards, and distributed locking — dramatically reducing load on backend databases.

### Tiers

| Tier | Memory | Clustering | Geo-replication | Use case |
|---|---|---|---|---|
| **Basic** | 250 MB – 53 GB | No | No | Dev/test; no SLA |
| **Standard** | 250 MB – 53 GB | No | No | Production; primary + replica; 99.9% SLA |
| **Premium** | 6 GB – 1.2 TB | Yes (up to 10 shards) | Active geo-replication | High throughput, large datasets, persistence |
| **Enterprise** | 12 GB – 14 TB | Yes (Redis Cluster) | Active geo-replication | Mission-critical; Redis modules (Search, JSON) |
| **Enterprise Flash** | Up to 14 TB (NVMe+RAM) | Yes | Active geo-replication | Cost-effective for very large datasets |

### Core Redis data types

| Type | Description | Use case |
|---|---|---|
| **String** | Binary-safe byte sequence | Simple key-value cache, counters |
| **Hash** | Field-value map under one key | User profile objects |
| **List** | Ordered linked list (push/pop) | Message queues, recent activity feed |
| **Set** | Unordered unique members | Tags, unique visitors |
| **Sorted Set (ZSet)** | Members with scores, sorted | Leaderboards, rate limiting |
| **Bitmap** | Bit array on a string | Feature flags, user online status |
| **HyperLogLog** | Probabilistic cardinality estimate | Approximate unique count |
| **Stream** | Append-only log with consumer groups | Event sourcing, IoT telemetry |

### Common caching patterns

**Cache-Aside (Lazy Loading)**
```
1. Application checks cache → miss
2. Application reads from database
3. Application writes result to cache (with TTL)
4. Subsequent reads hit cache
```
Most common pattern — only data that is actually requested gets cached.

**Write-Through**
```
1. Application writes to cache
2. Cache synchronously writes to database
3. Reads always hit cache (never stale)
```
Cache is always consistent with the database but every write has double latency.

**Write-Behind (Write-Back)**
```
1. Application writes to cache
2. Cache asynchronously flushes to database
3. High write throughput; risk of data loss on cache failure
```

### TTL and eviction

Set a **TTL (Time To Live)** on keys to expire stale data automatically. When the cache is full, Redis uses an **eviction policy** to make room:

| Policy | Behaviour |
|---|---|
| `allkeys-lru` | Evict least recently used keys across all keys |
| `volatile-lru` | Evict least recently used keys that have a TTL set |
| `allkeys-lfu` | Evict least frequently used keys |
| `volatile-ttl` | Evict keys with the shortest TTL first |
| `noeviction` | Return error when memory is full (default) |

## Redis Advanced Features

### Persistence (Premium tier)

Redis is in-memory — by default, data is lost on restart. The Premium tier adds two persistence options:

| Option | Description | Recovery |
|---|---|---|
| **RDB (Redis Database Backup)** | Periodic snapshot to Blob Storage | Restore from last snapshot — some data loss |
| **AOF (Append-Only File)** | Log every write operation | Near-zero data loss; higher overhead |

Use RDB for periodic backups with acceptable data loss. Use AOF when data durability is critical.

### Clustering (Premium tier)

Redis Cluster shards data across up to **10 shards** (Premium) or unlimited shards (Enterprise). Each shard has a primary and one or more replicas. Clustering increases:
- Total memory capacity
- Total throughput (commands per second)

Keys are distributed using **hash slots** (16,384 total). Client libraries handle slot routing automatically.

### Geo-replication

| Type | Tier | Behaviour |
|---|---|---|
| **Passive geo-replication** | Premium | One writable primary; secondary is read-only; cross-region failover requires manual intervention |
| **Active geo-replication** | Enterprise | All replicas are writable; conflict resolution by Last-Writer-Wins; cross-region active-active |

### VNet integration and Private Link

- **Premium tier** — deploy inside a VNet subnet (VNet injection); accessible only via private IP
- **Standard/Basic** — use **Private Link** to assign a private endpoint

Always disable public network access and use private connectivity in production.

### Redis as a distributed session store

Store ASP.NET Core or other web framework sessions in Redis instead of in-memory or a database:
```
User request → App Server A reads session from Redis
User request → App Server B reads same session from Redis  ← works with multiple instances
```
Allows horizontal scaling of stateless web servers without sticky sessions.

### Pub/Sub

Redis pub/sub allows producers to publish messages to **channels** and consumers to subscribe:
- Fire-and-forget — no message persistence, no consumer group tracking
- For durable messaging use **Redis Streams** (append-only log with consumer groups)
- For enterprise messaging use Azure Service Bus

## Azure Database for PostgreSQL

**Azure Database for PostgreSQL** is a fully managed service running the community PostgreSQL engine. Azure handles patching, backups, HA, and monitoring.

### Deployment options

| Option | Description | Use case |
|---|---|---|
| **Flexible Server** | Current recommended option; full control over maintenance window, stop/start, zone-redundant HA | All new PostgreSQL workloads |
| **Single Server** | Legacy — being retired in 2025; limited configuration | Migrate to Flexible Server |

### Flexible Server key features

| Feature | Description |
|---|---|
| **Zone-redundant HA** | Standby replica in a different AZ; automatic failover in ~60–120 seconds |
| **Same-zone HA** | Standby in same AZ — lower cost, no cross-zone latency |
| **Read replicas** | Up to 5 cross-region read replicas for read scale-out or DR |
| **Stop/start** | Stop server to save cost; billing stops for compute (storage still billed) |
| **PITR** | Point-in-time restore within 1–35 days |
| **Burstable tier** | B-series VMs — cost-effective for dev/test and intermittent workloads |
| **Extensions** | pg_partman, PostGIS, pgcrypto, pg_stat_statements, uuid-ossp, and more |
| **PgBouncer** | Built-in connection pooling — reduces connection overhead for high-concurrency apps |

### Compute tiers

| Tier | Use case |
|---|---|
| **Burstable** | Dev/test; intermittent CPU usage |
| **General Purpose** | Most production workloads |
| **Memory Optimised** | High-memory workloads — analytics, large working sets |

## Azure Database for MySQL and MariaDB

### Azure Database for MySQL — Flexible Server

Fully managed MySQL community edition. Identical architecture to PostgreSQL Flexible Server:

| Feature | Details |
|---|---|
| **HA** | Zone-redundant or same-zone standby; automatic failover |
| **Read replicas** | Up to 10 cross-region replicas |
| **Stop/start** | Stop server to pause compute billing |
| **PITR** | 1–35 days |
| **Versions** | MySQL 5.7 and 8.0 |

Suitable for LAMP stack applications and any workload built on MySQL.

### Azure Database for MariaDB

Managed MariaDB community edition — a MySQL-compatible fork. **Note:** Azure Database for MariaDB is being **retired on September 19, 2025**. Microsoft recommends migrating to Azure Database for MySQL Flexible Server.

### Common features across all open-source managed databases

| Feature | PostgreSQL | MySQL | MariaDB |
|---|---|---|---|
| Entra ID authentication | ✅ | ✅ | ✅ |
| Private endpoint / VNet | ✅ | ✅ | ✅ |
| TDE (at rest encryption) | ✅ | ✅ | ✅ |
| SSL enforcement | ✅ | ✅ | ✅ |
| Automated backups + PITR | ✅ | ✅ | ✅ |
| Read replicas | ✅ | ✅ | ✅ |
| Microsoft Defender | ✅ | ✅ | ✅ |

## Choosing the Right Database Service

| Scenario | Service |
|---|---|
| Sub-millisecond reads, reduce DB load | **Azure Cache for Redis** |
| Session state across multiple web servers | **Azure Cache for Redis** |
| Leaderboards, rate limiting, pub/sub | **Azure Cache for Redis** |
| Very large in-memory dataset (TB scale) | **Redis Enterprise Flash** |
| New PostgreSQL workload | **Azure Database for PostgreSQL Flexible Server** |
| LAMP stack / existing MySQL app | **Azure Database for MySQL Flexible Server** |
| Existing MariaDB app | Migrate to **MySQL Flexible Server** (MariaDB retiring 2025) |
| Relational + JSON + full text search | **PostgreSQL** with jsonb + pg_trgm extensions |
| Globally distributed low-latency reads | **Cosmos DB** (not relational — use only if schema flexibility is also needed) |

## Working with Azure Cache for Redis and PostgreSQL via Python

In [ ]:
# pip install azure-identity azure-mgmt-redis redis azure-mgmt-rdbms-postgresql psycopg2-binary

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.redis import RedisManagementClient
from azure.mgmt.redis.models import RedisCreateParameters, Sku

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"
location        = "eastus"

redis_mgmt = RedisManagementClient(credential, subscription_id)

In [ ]:
# Create a Standard C1 (1 GB) Redis cache
poller = redis_mgmt.redis.begin_create(
    resource_group, "my-redis-cache",
    RedisCreateParameters(
        location=location,
        sku=Sku(name="Standard", family="C", capacity=1),
        enable_non_ssl_port=False,       # enforce TLS only
        minimum_tls_version="1.2",
        redis_configuration={
            "maxmemory-policy": "allkeys-lru"  # evict LRU keys when full
        }
    )
)
print("Redis cache provisioning started — takes ~15 minutes")
cache = poller.result()
print(f"Cache: {cache.name}  host: {cache.host_name}  port: {cache.ssl_port}")

In [ ]:
# Get the primary access key
keys = redis_mgmt.redis.list_keys(resource_group, "my-redis-cache")
print(f"Primary key: {keys.primary_key[:8]}...  (truncated)")

# List all caches in resource group
for c in redis_mgmt.redis.list_by_resource_group(resource_group):
    print(f"  {c.name:<30} {c.sku.name} {c.sku.capacity}  {c.host_name}")

In [ ]:
import redis, json
from datetime import timedelta

# Connect to Azure Cache for Redis over TLS
r = redis.Redis(
    host="my-redis-cache.redis.cache.windows.net",
    port=6380,
    password=keys.primary_key,
    ssl=True
)
r.ping()  # verify connection
print("Connected to Redis")

In [ ]:
# --- String: simple key-value cache with TTL ---
r.setex("user:1001:name", timedelta(hours=1), "Alice")
print(f"Name: {r.get('user:1001:name').decode()}")

# --- Hash: store an object ---
r.hset("user:1001", mapping={"name": "Alice", "email": "alice@example.com", "plan": "premium"})
r.expire("user:1001", 3600)
user = r.hgetall("user:1001")
print({k.decode(): v.decode() for k, v in user.items()})

# --- Counter (atomic increment) ---
r.incr("pageviews:home")
r.incr("pageviews:home")
print(f"Page views: {r.get('pageviews:home').decode()}")

# --- Sorted set: leaderboard ---
r.zadd("leaderboard", {"alice": 9500, "bob": 8700, "carol": 9200})
top3 = r.zrevrange("leaderboard", 0, 2, withscores=True)
print("Top 3:", [(name.decode(), score) for name, score in top3])

# --- Cache-aside pattern ---
def get_product(product_id: str) -> dict:
    cache_key = f"product:{product_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)  # cache hit
    # Cache miss — fetch from DB (simulated)
    product = {"id": product_id, "name": "Widget", "price": 9.99}
    r.setex(cache_key, timedelta(minutes=15), json.dumps(product))
    return product

print(get_product("prod-001"))

In [ ]:
# Pipeline: batch multiple commands in one round trip
pipe = r.pipeline()
for i in range(100):
    pipe.setex(f"session:{i}", 1800, f"data-{i}")
pipe.execute()
print("Wrote 100 sessions in one pipeline")

# Pub/Sub example (producer side)
r.publish("notifications", json.dumps({"type": "order_shipped", "order_id": "ord-42"}))
print("Message published to notifications channel")

In [ ]:
from azure.mgmt.postgresqlflexibleservers import PostgreSQLManagementClient
from azure.mgmt.postgresqlflexibleservers.models import Server, ServerVersion, Sku, Storage, Backup, HighAvailability

pg = PostgreSQLManagementClient(credential, subscription_id)

In [ ]:
# Create a PostgreSQL Flexible Server with zone-redundant HA
poller = pg.servers.begin_create(
    resource_group, "my-pg-server",
    Server(
        location=location,
        version=ServerVersion.SIXTEEN,
        administrator_login="pgadmin",
        administrator_login_password="<strong-password>",
        sku=Sku(name="Standard_D2ds_v4", tier="GeneralPurpose"),
        storage=Storage(storage_size_gb=128),
        backup=Backup(backup_retention_days=14, geo_redundant_backup="Enabled"),
        high_availability=HighAvailability(
            mode="ZoneRedundant"   # standby replica in a different AZ
        )
    )
)
print("PostgreSQL Flexible Server provisioning started")
server = poller.result()
print(f"Server: {server.name}  FQDN: {server.fully_qualified_domain_name}")

In [ ]:
import psycopg2

# Connect using password (use Entra ID token for production)
conn = psycopg2.connect(
    host="my-pg-server.postgres.database.azure.com",
    database="postgres",
    user="pgadmin",
    password="<password>",
    sslmode="require"   # TLS required
)
cursor = conn.cursor()

# Create a table and insert rows
cursor.execute("""
    CREATE TABLE IF NOT EXISTS products (
        id   SERIAL PRIMARY KEY,
        name VARCHAR(100),
        price DECIMAL(10,2)
    )
""")
cursor.execute("INSERT INTO products (name, price) VALUES (%s, %s)", ("Widget", 9.99))
conn.commit()

cursor.execute("SELECT id, name, price FROM products")
for row in cursor.fetchall():
    print(f"  id={row[0]}  name={row[1]}  price={row[2]}")

conn.close()

## Summary

| Concept | Key Takeaway |
|---|---|
| Azure Cache for Redis | Fully managed Redis; sub-millisecond in-memory reads |
| Basic tier | Dev/test; no SLA; no replication |
| Standard tier | Production; primary + replica; 99.9% SLA |
| Premium tier | Clustering (10 shards), VNet injection, persistence (RDB/AOF), passive geo-replication |
| Enterprise tier | Redis modules (Search, JSON, TimeSeries); active geo-replication; larger memory |
| Cache-aside | Most common pattern — app checks cache, on miss reads DB and writes to cache |
| Write-through | App writes to cache; cache writes to DB — always consistent, double write latency |
| TTL | Set expiry on keys to prevent stale data |
| Eviction policy | allkeys-lru most common for caches; noeviction default (returns error when full) |
| RDB persistence | Periodic snapshots — small data loss on failure |
| AOF persistence | Log every write — near-zero data loss; higher overhead |
| Sorted set | Leaderboards and rate limiting — O(log N) insert, rank queries |
| Pipeline | Batch commands in one network round trip — dramatically reduces latency |
| Redis Streams | Durable pub/sub with consumer groups — prefer over basic pub/sub for reliability |
| PostgreSQL Flexible Server | Current recommended option; zone-redundant HA; stop/start; PgBouncer; read replicas |
| MySQL Flexible Server | Zone-redundant HA; stop/start; up to 10 read replicas |
| MariaDB | Being retired September 2025 — migrate to MySQL Flexible Server |
| VNet / Private endpoint | Always use private connectivity for production databases and caches |